In [65]:
import pandas as pd
import numpy as np
import re
DATASET = pd.read_csv('Comune-di-Milano-Servizi-alla-persona-parrucchieri-estetisti(in).csv',sep=';',encoding='unicode_escape')
DATASET = DATASET.fillna(pd.NA)
DATASET.head()

,Tipo esercizio pa,Ubicazione,Tipo via,Via,Civico,Codice via,ZD,Prevalente,Superficie altri usi,Superficie lavorativa
0,<NA>,LGO DEI GELSOMINI N. 10 (z.d. 6),LGO,DEI GELSOMINI,10,5394.0,6,<NA>,NaN,55.0
1,<NA>,PZA FIDIA N. 3 (z.d. 9),PZA,FIDIA,3,1144.0,9,CENTRO MASSAGGI RILASSANTI NON ESTETICI,2.0,28.0
2,<NA>,VIA ADIGE N. 10 (z.d. 5),VIA,ADIGE,10,4216.0,5,CENTRO BENESSERE,2.0,27.0
3,<NA>,VIA BARACCHINI FLAVIO N. 9 (z.d. 1),VIA,BARACCHINI FLAVIO,9,356.0,1,TRUCCO SEMIPERMANENTE,NaN,NaN
4,<NA>,VIA BERGAMO N. 12 (z.d. 4),VIA,BERGAMO,12,3189.0,4,<NA>,NaN,50.0


In [66]:
# --- IGNORE ---
# Righe che contengono "n." seguito da un numero (n.1, n.2, ecc.)
DATASET[DATASET['Ubicazione'].str.contains(r'n\.\s*\d+', na=False, regex=True)]

,Tipo esercizio pa,Ubicazione,Tipo via,Via,Civico,Codice via,ZD,Prevalente,Superficie altri usi,Superficie lavorativa
403,ACCONCIATORE,VIA DELLE FORZE ARMATE N. 410 ang. via ceriani...,VIA,DELLE FORZE ARMATE,410,6643.0,7,<NA>,4.0,32.0
1060,Centro massaggi,VIA TARTINI GIUSEPPE n. 34 (z.d. 9),VIA,FIUGGI,44,1328.0,9,<NA>,NaN,NaN
1086,Centro massaggi,VLE RODI N. 91 via rodi n. 91 (z.d. 9),VLE,RODI,91,1623.0,9,CENTRO MASSAGGI RILASSANTI NON ESTETICI,NaN,NaN
3090,TIPO A ESTETICA MANUALE;TIPO C TRATT.ESTETICI ...,VIA FERRIERI ENZO N. 12 ferrieri n.6; (z.d. 7),VIA,FERRIERI ENZO,12,6367.0,7,<NA>,NaN,NaN
3208,TIPO A - REG.2003,PLE ISTRIA N. 2 con ngresso v.le zara n. 132; ...,PLE,ISTRIA,2,1347.0,2,<NA>,NaN,NaN
3568,TIPO A - REG.2003;TIPO B CENTRO DI ABBRONZATURA,VIA AGNELLO con ingr. in via san paolo n.7 num...,VIA,VOLTA ALESSANDRO,8,1018.0,1,<NA>,NaN,NaN
3606,TIPO A - REG.2003;TIPO B CENTRO DI ABBRONZATURA,VIA CAVEZZALI FRANCESCO N. 11 con ingr. via ma...,VIA,CAVEZZALI FRANCESCO,11,2335.0,2,<NA>,NaN,NaN


In [67]:
#Normalization for Ubicazione
DATASET['Ubicazione'] = DATASET['Ubicazione'].str.replace('num', 'N')
DATASET['Ubicazione'] = DATASET['Ubicazione'].str.replace('n.', 'N.')

In [68]:
DATASET.drop(['Tipo esercizio pa', 'Prevalente', 'Superficie altri usi', 'Superficie lavorativa'], axis=1, inplace=True)
DATASET = DATASET[['Civico', 'ZD', 'Tipo via', 'Via', 'Codice via', 'Ubicazione']]

# Splitting the "Ubicazione" column on "N." to separate address and addition data
split_column = DATASET['Ubicazione'].str.split(r"\bN\.", n=1, expand=True)
split_column.columns = ['Ubicazione_effettiva', 'Ubicazione_data']

# Clean the Ubicazione_data by stripping leading/trailing whitespace
split_column['Ubicazione_data'] = split_column['Ubicazione_data'].str.strip()

# Add split columns to DATASET
DATASET = pd.concat([DATASET, split_column], axis=1)

# Splitting the "Ubicazione_data" column on "(" to separate civico and z.d.
split_column_2 = DATASET['Ubicazione_data'].str.split(r"\(z\.d\.", n=1, expand=True)
split_column_2.columns = ['Civico_to_check', 'ZD_to_check']

# Add to original database
DATASET = pd.concat([DATASET, split_column_2], axis=1)

# Remove ";" from "Civico_to_check"
DATASET['Civico_to_check'] = DATASET['Civico_to_check'].str.replace(";", "", regex=False)
DATASET['Civico_to_check'] = DATASET['Civico_to_check'].str.strip()

# Remove the closing parenthesis ")" and the ZD description from the "ZD_to_check" column
DATASET["ZD_to_check"] = DATASET["ZD_to_check"].str.replace("z.d.", "", regex=False)
DATASET["ZD_to_check"] = DATASET["ZD_to_check"].str.replace(")", "", regex=False)
DATASET['ZD_to_check'] = DATASET['ZD_to_check'].str.strip()

# Convert null/empty values to pd.NA
DATASET['Civico_to_check'] = DATASET['Civico_to_check'].fillna(pd.NA)
DATASET['ZD_to_check'] = DATASET['ZD_to_check'].fillna(pd.NA)

# Splitting the "Ubicazione_effettiva" column on the first " " to separate "Tipo via" and "Via"
split_column_3 = DATASET['Ubicazione_effettiva'].str.split(" ", n=1, expand=True)
split_column_3.columns = ['Tipo via_to_check', 'Via_to_check']

# Add to original database
DATASET = pd.concat([DATASET, split_column_3], axis=1)

# Convert null/empty values to pd.NA for Tipo via and Via
DATASET['Tipo via_to_check'] = DATASET['Tipo via_to_check'].fillna(pd.NA)
DATASET['Tipo via_to_check'] = DATASET['Tipo via_to_check'].str.strip()
DATASET['Via_to_check'] = DATASET['Via_to_check'].fillna(pd.NA)
DATASET['Via_to_check'] = DATASET['Via_to_check'].str.strip()

DATASET = DATASET.drop(['Ubicazione_effettiva', 'Ubicazione_data'], axis=1)
dfvia = DATASET
DATASET

,Civico,ZD,Tipo via,Via,Codice via,Ubicazione,Civico_to_check,ZD_to_check,Tipo via_to_check,Via_to_check
0,10,6,LGO,DEI GELSOMINI,5394.0,LGO DEI GELSOMINI N. 10 (z.d. 6),10,6,LGO,DEI GELSOMINI
1,3,9,PZA,FIDIA,1144.0,PZA FIDIA N. 3 (z.d. 9),3,9,PZA,FIDIA
2,10,5,VIA,ADIGE,4216.0,VIA ADIGE N. 10 (z.d. 5),10,5,VIA,ADIGE
3,9,1,VIA,BARACCHINI FLAVIO,356.0,VIA BARACCHINI FLAVIO N. 9 (z.d. 1),9,1,VIA,BARACCHINI FLAVIO
4,12,4,VIA,BERGAMO,3189.0,VIA BERGAMO N. 12 (z.d. 4),12,4,VIA,BERGAMO
...,...,...,...,...,...,...,...,...,...,...
3904,1,1,VIA,SARPI FRA' PAOLO,7210.0,VIA SARPI FRA' PAOLO N. 1 con ingr.da v.le mon...,1 con ingr.da v.le montello 4/6,1,VIA,SARPI FRA' PAOLO
3905,4,1,CSO,DI PORTA TICINESE,541.0,CSO DI PORTA TICINESE N. 4 ; (z.d. 1),4,1,CSO,DI PORTA TICINESE
3906,2,9,VIA,CANDOGLIA,1518.0,VIA CANDOGLIA N. 2 ; (z.d. 9),2,9,VIA,CANDOGLIA
3907,<NA>,1,VIA,NIRONE,640.0,VIA NIRONE N.002a; (z.d. 1),002a,1,VIA,NIRONE


### Civico e ZD

In [69]:
# Check if the civico is equal to the civico_to_check, return only the rows where they are not equal
notok = DATASET[(DATASET['Civico'] != DATASET['Civico_to_check']) | (DATASET['ZD'] != DATASET['ZD_to_check'])]
notok

,Civico,ZD,Tipo via,Via,Codice via,Ubicazione,Civico_to_check,ZD_to_check,Tipo via_to_check,Via_to_check
20,23,3,VIA,STRAMBIO GAETANO,3168.0,VIA STRAMBIO GAETANO N. 23 estetista in appart...,23 estetista in appartamento,3,VIA,STRAMBIO GAETANO
32,<NA>,<NA>,<NA>,<NA>,NaN,CSO COMO N. 15 interno club f. conti,15 interno club f. conti,<NA>,CSO,COMO
33,1111,ACCONCIATORE,COMO,15,9.0,CSO,<NA>,<NA>,CSO,<NA>
45,2,1,PAS,DUOMO,102.0,PAS DUOMO N. 2 piano primo; (z.d. 1),2 piano primo,1,PAS,DUOMO
92,17,6,VIA,DARWIN CARLO ROBERTO,5238.0,VIA DARWIN CARLO ROBERTO N. 17 int. residenze ...,17 int. residenze anni azzurri (per anziani),6,VIA,DARWIN CARLO ROBERTO
...,...,...,...,...,...,...,...,...,...,...
3890,48,6,VIA,GESSI ROMOLO,6121.0,VIA GESSI ROMOLO N. 48 con ingresso v.fra' bar...,48 con ingresso v.fra' bartolomeo,6,VIA,GESSI ROMOLO
3896,4,7,VIA,DURER ALBERTO,6602.0,VIA DURER ALBERTO N. 4 primo piano; (z.d. 7),4 primo piano,7,VIA,DURER ALBERTO
3897,31,2,VIA,ZURETTI GIANFRANCO,1216.0,VIA ZURETTI GIANFRANCO N. 31 al secondo piano ...,31 al secondo piano scala a,2,VIA,ZURETTI GIANFRANCO
3904,1,1,VIA,SARPI FRA' PAOLO,7210.0,VIA SARPI FRA' PAOLO N. 1 con ingr.da v.le mon...,1 con ingr.da v.le montello 4/6,1,VIA,SARPI FRA' PAOLO


In [70]:
# Clean and convert Civico_to_check and ZD_to_check before comparison
#def clean_numeric_string(value):
#    if pd.isna(value):
#        return pd.NA, pd.NA
#    
#    if isinstance(value, str):
#        value_str = value.strip()
#        if not value_str:
#            return pd.NA, pd.NA
#        
#        # Check if value contains at least one digit, otherwise return pd.NA
#        if not re.search(r'\d', value_str):
#            return pd.NA, pd.NA
#        
#        # Split after the first number
#        match = re.match(r'^(\d+(?:[a-zA-Z](?![a-zA-Z])|/\d*)?)\s*(.*)', value_str)
#        
#        if match:
#            numeric_part = match.group(1)
#            additional_part = match.group(2) if match.group(2) else pd.NA
#        else:
#            # Fallback: if no match, return pd.NA
#            return pd.NA, value
#        
#        # Remove leading zeros but keep at least one digit, preserving suffixes like /2, a, b, etc.
#        numeric_part = re.sub(r'^0+(?=\d)', '', numeric_part)
#        if not numeric_part:
#            numeric_part = "0"
#        
#        return numeric_part, additional_part
#        
#    elif isinstance(value, (int, float)):
#        return str(int(value)), pd.NA
#        return str(int(value)), pd.NA
#    
#    return pd.NA, pd.NA

# Clean and convert Civico_to_check and ZD_to_check before comparison
def clean_numeric_string(value):
    if pd.isna(value):
        return pd.NA, pd.NA
    
    if isinstance(value, str):
        value_str = value.strip()
        if not value_str:
            return pd.NA, pd.NA
        
        # Check if value contains at least one digit, otherwise return pd.NA
        if not re.search(r'\d', value_str):
            return pd.NA, pd.NA
        
        # Find all numeric sequences with optional suffixes (like 7, 0061, 22c, 6/2)
        all_numbers = re.findall(r'\d+(?:[a-zA-Z](?![a-zA-Z])|/\d*)?', value_str)
        
        if not all_numbers:
            return pd.NA, value
        
        # Look for numbers with leading zeros (e.g., 0061, 008, 002a)
        numbers_with_leading_zeros = [num for num in all_numbers if re.match(r'^0\d+', num)]
        
        # Prioritize numbers with leading zeros, otherwise take the first number
        if numbers_with_leading_zeros:
            numeric_part = numbers_with_leading_zeros[0]
        else:
            numeric_part = all_numbers[0]
        
        # Find the position of the selected number and extract remaining text after it
        num_position = value_str.find(numeric_part)
        remaining = value_str[num_position + len(numeric_part):].strip()
        additional_part = remaining if remaining else pd.NA
        
        # Remove leading zeros but keep at least one digit, preserving suffixes like /2, a, b, etc.
        numeric_part = re.sub(r'^0+(?=\d)', '', numeric_part)
        if not numeric_part:
            numeric_part = "0"
        
        return numeric_part, additional_part
        
    elif isinstance(value, (int, float)):
        return str(int(value)), pd.NA
    
    return pd.NA, pd.NA

# Apply cleaning to Civico_to_check and save both parts
DATASET[['Civico_to_check', 'Civico_additional']] = DATASET['Civico_to_check'].apply(
    lambda x: pd.Series(clean_numeric_string(x))
)
DATASET[['ZD_to_check', 'ZD_additional']] = DATASET['ZD_to_check'].apply(
    lambda x: pd.Series(clean_numeric_string(x))
)

# Convert to string for comparison, replacing pd.NA with empty string
DATASET['Civico'] = DATASET['Civico'].astype(str).replace('<NA>', '0').replace('nan', '0')
DATASET['ZD'] = DATASET['ZD'].astype(str).replace('<NA>', '0').replace('nan', '0')
DATASET['Civico_to_check'] = DATASET['Civico_to_check'].astype(str).replace('<NA>', '0')
DATASET['ZD_to_check'] = DATASET['ZD_to_check'].astype(str).replace('<NA>', '0')

# Check the not right values
condition = ((DATASET['Civico_to_check'] != DATASET['Civico']) | (DATASET['ZD_to_check'] != DATASET['ZD']))
check = DATASET[condition][['Civico', 'Civico_to_check', 'Civico_additional', 'ZD', 'ZD_to_check', 'ZD_additional']]
check

,Civico,Civico_to_check,Civico_additional,ZD,ZD_to_check,ZD_additional
32,0,15,interno club f. conti,0,0,<NA>
33,1111,0,<NA>,ACCONCIATORE,0,<NA>
144,40945,6/2,<NA>,5,5,<NA>
210,72,8,<NA>,4,4,<NA>
243,10,0,<NA>,1,1,<NA>
...,...,...,...,...,...,...
3767,157,43,<NA>,6,6,<NA>
3801,0,21a,<NA>,8,8,<NA>
3818,13,21a,<NA>,5,5,<NA>
3839,0,11d,<NA>,6,6,<NA>


In [71]:
# If Civico_to_check is not null and is not equal to Civico, replace Civico with Civico_to_check
DATASET.loc[:, 'Civico'] = DATASET.apply(
    lambda row: row["Civico_to_check"] if row["Civico"] == 0 and row["Civico_to_check"] != 0 else 
                (row["Civico"] if row["Civico_to_check"] == 0 else 
                (row["Civico_to_check"] if row["Civico"] != row["Civico_to_check"] else row["Civico"])), axis=1)
# If ZD_to_check is not null and is not equal to ZD, replace ZD with ZD_to_check
DATASET.loc[:, 'ZD'] = DATASET.apply(
    lambda row: row["ZD_to_check"] if row["ZD"] == 0 and row["ZD_to_check"] != 0 else 
                (row["ZD"] if row["ZD_to_check"] == 0 else 
                (row["ZD_to_check"] if row["ZD"] != row["ZD_to_check"] else row["ZD"])), axis=1)

DATASET.iloc[check.index]

,Civico,ZD,Tipo via,Via,Codice via,Ubicazione,Civico_to_check,ZD_to_check,Tipo via_to_check,Via_to_check,Civico_additional,ZD_additional
32,15,0,<NA>,<NA>,NaN,CSO COMO N. 15 interno club f. conti,15,0,CSO,COMO,interno club f. conti,<NA>
33,0,0,COMO,15,9.0,CSO,0,0,CSO,<NA>,<NA>,<NA>
144,6/2,5,VIA,PAVIA,5262.0,VIA PAVIA N. 6/2 (z.d. 5),6/2,5,VIA,PAVIA,<NA>,<NA>
210,8,4,CSO,LODI,4068.0,codvia 4386 N.008; (z.d. 4),8,4,codvia,4386,<NA>,<NA>
243,0,1,CSO,SEMPIONE,7137.0,mm1 duomo codvia 9112 N.000; (z.d. 1),0,1,mm1,duomo codvia 9112,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...
3767,43,6,VIA,LORENTEGGIO,5132.0,VIA SOLARI ANDREA N.043; (z.d. 6),43,6,VIA,SOLARI ANDREA,<NA>,<NA>
3801,21a,8,VLE,CERTOSA,7174.0,VLE CERTOSA interno palestra N.021a; (z.d. 8),21a,8,VLE,CERTOSA interno palestra,<NA>,<NA>
3818,21a,5,VIA,LAGRANGE GIUSEPPE,5205.0,VLE SABOTINO N.021a; (z.d. 5),21a,5,VLE,SABOTINO,<NA>,<NA>
3839,11d,6,VIA,BARI,5218.0,VIA BARI N.011d; (z.d. 6),11d,6,VIA,BARI,<NA>,<NA>


Ci sono alcuni civici e ZD pari a 0

In [72]:
#Check all the rows with again different values
condition = ((DATASET['Civico_to_check'] != DATASET['Civico']) | (DATASET['ZD_to_check'] != DATASET['ZD']))
check = DATASET[condition]
check

,Civico,ZD,Tipo via,Via,Codice via,Ubicazione,Civico_to_check,ZD_to_check,Tipo via_to_check,Via_to_check,Civico_additional,ZD_additional


### Tipo Via

In [73]:
dfvia = dfvia.drop(['Civico_to_check', 'ZD_to_check', 'Civico_additional', 'ZD_additional'], axis=1)
dfvia

,Civico,ZD,Tipo via,Via,Codice via,Ubicazione,Tipo via_to_check,Via_to_check
0,10,6,LGO,DEI GELSOMINI,5394.0,LGO DEI GELSOMINI N. 10 (z.d. 6),LGO,DEI GELSOMINI
1,3,9,PZA,FIDIA,1144.0,PZA FIDIA N. 3 (z.d. 9),PZA,FIDIA
2,10,5,VIA,ADIGE,4216.0,VIA ADIGE N. 10 (z.d. 5),VIA,ADIGE
3,9,1,VIA,BARACCHINI FLAVIO,356.0,VIA BARACCHINI FLAVIO N. 9 (z.d. 1),VIA,BARACCHINI FLAVIO
4,12,4,VIA,BERGAMO,3189.0,VIA BERGAMO N. 12 (z.d. 4),VIA,BERGAMO
...,...,...,...,...,...,...,...,...
3904,1,1,VIA,SARPI FRA' PAOLO,7210.0,VIA SARPI FRA' PAOLO N. 1 con ingr.da v.le mon...,VIA,SARPI FRA' PAOLO
3905,4,1,CSO,DI PORTA TICINESE,541.0,CSO DI PORTA TICINESE N. 4 ; (z.d. 1),CSO,DI PORTA TICINESE
3906,2,9,VIA,CANDOGLIA,1518.0,VIA CANDOGLIA N. 2 ; (z.d. 9),VIA,CANDOGLIA
3907,2a,1,VIA,NIRONE,640.0,VIA NIRONE N.002a; (z.d. 1),VIA,NIRONE


In [74]:
# Check if there are some differences between Tipo via and Tipo via_to_check, Via and Via_to_check
condition = (
    (dfvia['Tipo via'].fillna(' ') != dfvia['Tipo via_to_check'].fillna(' ')) 
)
check = dfvia[condition]
check

,Civico,ZD,Tipo via,Via,Codice via,Ubicazione,Tipo via_to_check,Via_to_check
32,15,0,<NA>,<NA>,NaN,CSO COMO N. 15 interno club f. conti,CSO,COMO
33,0,0,COMO,15,9.0,CSO,CSO,<NA>
210,8,4,CSO,LODI,4068.0,codvia 4386 N.008; (z.d. 4),codvia,4386
243,0,1,CSO,SEMPIONE,7137.0,mm1 duomo codvia 9112 N.000; (z.d. 1),mm1,duomo codvia 9112
651,2a,6,PZA,FRATTINI PIETRO,5389.0,VIA SOLARI ANDREA N.002a; (z.d. 6),VIA,SOLARI ANDREA
744,509,4,VIA,MINCIO,4157.0,VLE FORLANINI ENRICO con ingr. da p.zza artigi...,VLE,FORLANINI ENRICO con ingr. da p.zza artigianato
745,505,4,VIA,TAGLIAMENTO,4147.0,VLE FORLANINI ENRICO N.0505; (z.d. 4),VLE,FORLANINI ENRICO
808,51a,1,VLE,CRISPI FRANCESCO,1063.0,CSO GARIBALDI GIUSEPPE N.051a; (z.d. 1),CSO,GARIBALDI GIUSEPPE
1203,192,8,VIA,RUGGERO DI LAURIA,7163.0,VLE TEODORICO N.0192; (z.d. 8),VLE,TEODORICO
1633,55a,2,PZA,QUATTRO NOVEMBRE,1199.0,VIA GIOIA MELCHIORRE N.055a; (z.d. 2),VIA,GIOIA MELCHIORRE


In [75]:
# Save in csv file all the types of Tipo Via
DATASET['Tipo via'].drop_duplicates().dropna().to_csv('tipo_via_types.csv', index=False)

In [76]:
# Check if the Tipo via_to_check is in the dataset of valid Tipo via types
valid_tipo_via = pd.read_csv('tipo_via_types.csv')['Tipo via'].tolist()

# Convert to string first to avoid pd.NA comparison issues
dfvia['Tipo via'] = dfvia['Tipo via'].astype(str).replace('<NA>', ' ').replace('nan', ' ')
dfvia['Tipo via_to_check'] = dfvia['Tipo via_to_check'].astype(str).replace('<NA>', ' ').replace('nan', ' ')

dfvia['Tipo via_to_check'] = dfvia['Tipo via_to_check'].apply(
    lambda x: x if x in valid_tipo_via else ' '
)

# Substitute Tipo via with Tipo via_to_check if they are different and Tipo via_to_check is not null
dfvia.loc[:, 'Tipo via'] = dfvia.apply(
    lambda row: row['Tipo via_to_check'] if row['Tipo via_to_check'] != ' ' and row['Tipo via'] != row['Tipo via_to_check'] else row['Tipo via'],
    axis=1
)

dfvia.iloc[check.index]

,Civico,ZD,Tipo via,Via,Codice via,Ubicazione,Tipo via_to_check,Via_to_check
32,15,0,CSO,<NA>,NaN,CSO COMO N. 15 interno club f. conti,CSO,COMO
33,0,0,CSO,15,9.0,CSO,CSO,<NA>
210,8,4,CSO,LODI,4068.0,codvia 4386 N.008; (z.d. 4),,4386
243,0,1,CSO,SEMPIONE,7137.0,mm1 duomo codvia 9112 N.000; (z.d. 1),,duomo codvia 9112
651,2a,6,VIA,FRATTINI PIETRO,5389.0,VIA SOLARI ANDREA N.002a; (z.d. 6),VIA,SOLARI ANDREA
744,509,4,VLE,MINCIO,4157.0,VLE FORLANINI ENRICO con ingr. da p.zza artigi...,VLE,FORLANINI ENRICO con ingr. da p.zza artigianato
745,505,4,VLE,TAGLIAMENTO,4147.0,VLE FORLANINI ENRICO N.0505; (z.d. 4),VLE,FORLANINI ENRICO
808,51a,1,CSO,CRISPI FRANCESCO,1063.0,CSO GARIBALDI GIUSEPPE N.051a; (z.d. 1),CSO,GARIBALDI GIUSEPPE
1203,192,8,VLE,RUGGERO DI LAURIA,7163.0,VLE TEODORICO N.0192; (z.d. 8),VLE,TEODORICO
1633,55a,2,VIA,QUATTRO NOVEMBRE,1199.0,VIA GIOIA MELCHIORRE N.055a; (z.d. 2),VIA,GIOIA MELCHIORRE


In [77]:
condition = (
    (dfvia['Tipo via'].fillna(' ') != dfvia['Tipo via_to_check'].fillna(' ')) 
)
check = dfvia[condition]
check

,Civico,ZD,Tipo via,Via,Codice via,Ubicazione,Tipo via_to_check,Via_to_check
210,8,4,CSO,LODI,4068.0,codvia 4386 N.008; (z.d. 4),,4386
243,0,1,CSO,SEMPIONE,7137.0,mm1 duomo codvia 9112 N.000; (z.d. 1),,duomo codvia 9112


### Via

In [78]:
dfvia = dfvia.drop(['Civico', 'ZD'], axis=1)

# Take the part in Via_to_check before the first minuscule letter or number
dfvia['Via_to_check'] = dfvia['Via_to_check'].str.split(r'[a-z0-9]', n=1).str[0].str.strip()
dfvia['Via'] = dfvia['Via'].astype(str).replace('<NA>', ' ').replace('nan', ' ').str.strip()
dfvia['Via_to_check'] = dfvia['Via_to_check'].astype(str).replace('<NA>', ' ').replace('nan', ' ').str.strip()

condition = (
    (dfvia['Via'].fillna(' ') != dfvia['Via_to_check'].fillna(' ')) 
)
check = dfvia[condition]
check

,Tipo via,Via,Codice via,Ubicazione,Tipo via_to_check,Via_to_check
32,CSO,,NaN,CSO COMO N. 15 interno club f. conti,CSO,COMO
33,CSO,15,9.0,CSO,CSO,
210,CSO,LODI,4068.0,codvia 4386 N.008; (z.d. 4),,
243,CSO,SEMPIONE,7137.0,mm1 duomo codvia 9112 N.000; (z.d. 1),,
319,VIA,FERMIGNANO,1607.0,VIA BOVISASCA N.173; (z.d. 9),VIA,BOVISASCA
...,...,...,...,...,...,...
3699,VIA,VENEZIA,238.0,VIA PAGANO MARIO piano rialzato N.03739; (z.d. 1),VIA,PAGANO MARIO
3713,VIA,GROSSICH ANTONIO,2576.0,VIA PILO ROSOLINO N.019b; (z.d. 3),VIA,PILO ROSOLINO
3764,VIA,MINCIO,4157.0,VIA SERLIO SEBASTIANO N.0082; (z.d. 4),VIA,SERLIO SEBASTIANO
3767,VIA,LORENTEGGIO,5132.0,VIA SOLARI ANDREA N.043; (z.d. 6),VIA,SOLARI ANDREA


In [79]:
# Substitute Via with Via_to_check if they are different and Via_to_check is not empty
dfvia.loc[:, 'Via'] = dfvia.apply(
    lambda row: row['Via_to_check'].strip() if row['Via_to_check'].strip() != '' and row['Via'].strip() != row['Via_to_check'].strip() else row['Via'].strip(),
    axis=1
)

dfvia.iloc[check.index]

,Tipo via,Via,Codice via,Ubicazione,Tipo via_to_check,Via_to_check
32,CSO,COMO,NaN,CSO COMO N. 15 interno club f. conti,CSO,COMO
33,CSO,15,9.0,CSO,CSO,
210,CSO,LODI,4068.0,codvia 4386 N.008; (z.d. 4),,
243,CSO,SEMPIONE,7137.0,mm1 duomo codvia 9112 N.000; (z.d. 1),,
319,VIA,BOVISASCA,1607.0,VIA BOVISASCA N.173; (z.d. 9),VIA,BOVISASCA
...,...,...,...,...,...,...
3699,VIA,PAGANO MARIO,238.0,VIA PAGANO MARIO piano rialzato N.03739; (z.d. 1),VIA,PAGANO MARIO
3713,VIA,PILO ROSOLINO,2576.0,VIA PILO ROSOLINO N.019b; (z.d. 3),VIA,PILO ROSOLINO
3764,VIA,SERLIO SEBASTIANO,4157.0,VIA SERLIO SEBASTIANO N.0082; (z.d. 4),VIA,SERLIO SEBASTIANO
3767,VIA,SOLARI ANDREA,5132.0,VIA SOLARI ANDREA N.043; (z.d. 6),VIA,SOLARI ANDREA


In [80]:
condition = (
    (dfvia['Via'].fillna(' ') != dfvia['Via_to_check'].fillna(' ')) 
)
check = dfvia[condition]
check

,Tipo via,Via,Codice via,Ubicazione,Tipo via_to_check,Via_to_check
33,CSO,15,9.0,CSO,CSO,
210,CSO,LODI,4068.0,codvia 4386 N.008; (z.d. 4),,
243,CSO,SEMPIONE,7137.0,mm1 duomo codvia 9112 N.000; (z.d. 1),,


### Codice via

In [81]:
# Save in csv file all the different Via with their corrispective codice via, without duplicates
# Also save the number of times each row appears
old_codice_via = DATASET[['Tipo via', 'Via', 'Codice via']].copy()
old_codice_via['count'] = old_codice_via.groupby(['Tipo via', 'Via', 'Codice via'])['Tipo via'].transform('count')
old_codice_via = old_codice_via.drop_duplicates()
old_codice_via = old_codice_via.sort_values(by=['Tipo via', 'Via'])
old_codice_via.to_csv('codice_via_mapping_pre.csv', index=False)

In [82]:
# Show rows where the same Tipo via + Via combination has multiple different Codice via values
duplicated_mask = old_codice_via.duplicated(subset=['Tipo via', 'Via'], keep=False)
old_codice_via[duplicated_mask].sort_values(by=['Tipo via', 'Via', 'Codice via']).to_csv('duplicated_tipo_via_via.csv', index=False)

# Save the mapping of (Tipo via, Via) that are new and not in the pre dataset
new_codice_via = dfvia[['Tipo via', 'Via', 'Codice via']].copy()
new_codice_via['count'] = new_codice_via.groupby(['Tipo via', 'Via', 'Codice via'])['Tipo via'].transform('count')
new_codice_via = new_codice_via.drop_duplicates()
new_codice_via = new_codice_via[~new_codice_via.set_index(['Tipo via', 'Via']).index.isin(old_codice_via.set_index(['Tipo via', 'Via']).index)]
new_codice_via = new_codice_via.sort_values(by=['Tipo via', 'Via'])
new_codice_via.to_csv('codice_via_mapping_not_in_pre.csv', index=False)
new_codice_via

,Tipo via,Via,Codice via,count
33,CSO,15,9.0,1.0
319,VIA,BOVISASCA,1607.0,1.0
3586,VIA,BOVISASCA,1109.0,1.0
1438,VIA,CALDERA,6194.0,1.0
2490,VIA,CICERONE MARCO TULLIO,1294.0,1.0
3311,VIA,F.LLI GABBA,314.0,1.0
1161,VIA,GNOCCHI DON CARLO,6100.0,1.0
3042,VIA,GNOCCHI DON CARLO,6212.0,1.0
1693,VIA,LONGARONE,7241.0,1.0
2845,VIA,LONGARONE,7292.0,1.0


In [83]:
# For the new mapping (Tipo via, Via) -> Set Codice via to 0
for index, row in new_codice_via.iterrows():
    tipo_via = row['Tipo via']
    via = row['Via']
    new_codice_via.loc[(new_codice_via['Tipo via'] == tipo_via) & (new_codice_via['Via'] == via), 'Codice via'] = 0

new_codice_via

,Tipo via,Via,Codice via,count
33,CSO,15,0.0,1.0
319,VIA,BOVISASCA,0.0,1.0
3586,VIA,BOVISASCA,0.0,1.0
1438,VIA,CALDERA,0.0,1.0
2490,VIA,CICERONE MARCO TULLIO,0.0,1.0
3311,VIA,F.LLI GABBA,0.0,1.0
1161,VIA,GNOCCHI DON CARLO,0.0,1.0
3042,VIA,GNOCCHI DON CARLO,0.0,1.0
1693,VIA,LONGARONE,0.0,1.0
2845,VIA,LONGARONE,0.0,1.0


Capire cosa fare per i codice via che hanno due valori diversi singoli e per quelli che non hanno un codice via iniziale

DA CONTROLLARE RIGA SEMPIONE (quella con mm1)

### Test

In [84]:
# Join all together
final = DATASET.copy()
# First, update DATASET with the modified Tipo via and Via from dfvia
final['Tipo via'] = dfvia['Tipo via']
final['Via'] = dfvia['Via']

# Create a mapping from not_in_pre with new Codice via values
not_in_pre_mapping = new_codice_via.set_index(['Tipo via', 'Via'])['Codice via'].to_dict()

# Update Codice via in final for rows that match not_in_pre
final['Codice via'] = final.apply(
    lambda row: not_in_pre_mapping.get((row['Tipo via'], row['Via']), row['Codice via']),
    axis=1
)

# For rows in final where Codice via is null, check if there's a match in pre and assign it
pre_mapping = old_codice_via.set_index(['Tipo via', 'Via'])['Codice via'].to_dict()
for index, row in final[final['Codice via'].isnull()].iterrows():
    key = (row['Tipo via'], row['Via'])
    if key in pre_mapping:
        final.at[index, 'Codice via'] = pre_mapping[key]

final

,Civico,ZD,Tipo via,Via,Codice via,Ubicazione,Civico_to_check,ZD_to_check,Tipo via_to_check,Via_to_check,Civico_additional,ZD_additional
0,10,6,LGO,DEI GELSOMINI,5394.0,LGO DEI GELSOMINI N. 10 (z.d. 6),10,6,LGO,DEI GELSOMINI,<NA>,<NA>
1,3,9,PZA,FIDIA,1144.0,PZA FIDIA N. 3 (z.d. 9),3,9,PZA,FIDIA,<NA>,<NA>
2,10,5,VIA,ADIGE,4216.0,VIA ADIGE N. 10 (z.d. 5),10,5,VIA,ADIGE,<NA>,<NA>
3,9,1,VIA,BARACCHINI FLAVIO,356.0,VIA BARACCHINI FLAVIO N. 9 (z.d. 1),9,1,VIA,BARACCHINI FLAVIO,<NA>,<NA>
4,12,4,VIA,BERGAMO,3189.0,VIA BERGAMO N. 12 (z.d. 4),12,4,VIA,BERGAMO,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...
3904,1,1,VIA,SARPI FRA' PAOLO,7210.0,VIA SARPI FRA' PAOLO N. 1 con ingr.da v.le mon...,1,1,VIA,SARPI FRA' PAOLO,con ingr.da v.le montello 4/6,<NA>
3905,4,1,CSO,DI PORTA TICINESE,541.0,CSO DI PORTA TICINESE N. 4 ; (z.d. 1),4,1,CSO,DI PORTA TICINESE,<NA>,<NA>
3906,2,9,VIA,CANDOGLIA,1518.0,VIA CANDOGLIA N. 2 ; (z.d. 9),2,9,VIA,CANDOGLIA,<NA>,<NA>
3907,2a,1,VIA,NIRONE,640.0,VIA NIRONE N.002a; (z.d. 1),2a,1,VIA,NIRONE,<NA>,<NA>


In [85]:
# print all rows that have different value
final['Via_to_check'] = final['Via_to_check'].str.split(r'[a-z0-9]', n=1).str[0].str.strip()
condition = (
    (final['Tipo via'].fillna(' ').str.strip() != final['Tipo via_to_check'].fillna(' ').str.strip()) |
    (final['Via'].fillna(' ').str.strip() != final['Via_to_check'].fillna(' ').str.strip()) |
    (final['Civico'].fillna(' ').str.strip() != final['Civico_to_check'].fillna(' ').str.strip()) |
    (final['ZD'].fillna(' ').str.strip() != final['ZD_to_check'].fillna(' ').str.strip())
)
final[condition]

,Civico,ZD,Tipo via,Via,Codice via,Ubicazione,Civico_to_check,ZD_to_check,Tipo via_to_check,Via_to_check,Civico_additional,ZD_additional
33,0,0,CSO,15,0.0,CSO,0,0,CSO,<NA>,<NA>,<NA>
210,8,4,CSO,LODI,4068.0,codvia 4386 N.008; (z.d. 4),8,4,codvia,,<NA>,<NA>
243,0,1,CSO,SEMPIONE,7137.0,mm1 duomo codvia 9112 N.000; (z.d. 1),0,1,mm1,,<NA>,<NA>


Ho cambiato tutto, ora vedo che ci sono dei numeri civici = 0, quindi non so, magari dobbiamo cambiare qualcosa. Anche ZD non può essere 0